# CIC-IDS2017 imbalance techniques experiment

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

RANDOM_STATE = 5

In [2]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)
def evaluate(preds, y, target_names, title=""):
    """Print report + F2, show confusion matrix. Returns the predictions."""
    print(f"--- {title} ---")
    print(f"accuracy: {(preds == y).mean():.4f}")
    print(classification_report(y, preds, target_names=target_names))
    #cm = confusion_matrix(y, preds) #labels=target_names)
    #ConfusionMatrixDisplay(cm).plot()
    #return preds

## Data Preparation

In [3]:
import glob
files = glob.glob("data/*.csv")

### Loading and Cleaning
First we must concatenate all 8 csv files. They have odd column names with trailing spaces, so names must be normalised. We then clean any rows that contain infinity or NaNs.

In [4]:
df_before_dropna = (
    pd.concat((pd.read_csv(f, encoding_errors="replace") for f in files), ignore_index=True)
    .rename(columns=lambda s: s.strip())
    .replace([np.inf, -np.inf], np.nan)
)

In [ ]:
dropped = df_before_dropna[df_before_dropna.isna().any(axis=1)]
print(dropped["Label"].value_counts())
df = df_before_dropna.dropna()
print(df["Label"].value_counts())

Dropping NaN values does not harm the model, we aren't dropping from rare classes.

### Down-Casting
Down-casting from float64 is inherantly lossy, so below checks if loss > 1e-07 when casting to float32. 

In [5]:
for col in df.select_dtypes("float").columns:
    orig = df[col].astype("float64")
    back = df[col].astype("float32").astype("float64")
    
    # largest relative error introduced
    rel_err = ((back - orig).abs() / orig.abs().replace(0, np.nan)).max()
    print(col, rel_err)

Fwd Packet Length Mean 5.8808532115861154e-08
Fwd Packet Length Std 5.946117860918782e-08
Bwd Packet Length Mean 5.894248777797077e-08
Bwd Packet Length Std 5.9490018992827366e-08
Flow Bytes/s 5.952342611128164e-08
Flow Packets/s 5.9561211665086304e-08
Flow IAT Mean 5.960429305881185e-08
Flow IAT Std 5.9494370761997505e-08
Fwd IAT Mean 5.9603710426339974e-08
Fwd IAT Std 5.952222580744429e-08
Bwd IAT Mean 5.9604513325274406e-08
Bwd IAT Std 5.958431058164412e-08
Fwd Packets/s 5.9561211665086304e-08
Bwd Packets/s 5.9527607936747004e-08
Packet Length Mean 5.9111707366382436e-08
Packet Length Std 5.9580811682697106e-08
Packet Length Variance 5.945458386220663e-08
Average Packet Size 5.8961517633826155e-08
Avg Fwd Segment Size 5.8808532115861154e-08
Avg Bwd Segment Size 5.894248777797077e-08
Active Mean 5.957727658327655e-08
Active Std 5.9495230822478896e-08
Idle Mean 5.960207627408825e-08
Idle Std 5.9463410133870844e-08


In [6]:
# Safely downcasts 64-bit data to 32-bit where there is no information loss
def down_cast(d):
    original_size = d.memory_usage(deep=True).sum()
    for col in d.select_dtypes("integer").columns:
        d[col] = pd.to_numeric(d[col], downcast="integer")
    for col in d.select_dtypes("float").columns:
        d[col] = pd.to_numeric(d[col], downcast="float")
    final_size = d.memory_usage(deep=True).sum()
    print("Size savings: " + str(original_size-final_size) + " bytes")
    return d

df = down_cast(df)

Size savings: 882297312 bytes


We save >0.88 GB for our precious, precious RAM. 

### De-duplication

In [7]:
original_size = df.shape[0]
df = df.drop_duplicates()
new_size = df.shape[0]
original_size-new_size

307084

CIC-IDS2017 contains a lot of duplicate rows (307,084!!). These must be removed to prevent leakage - a row could be in the training set and then it's twin could appear in the test set. Now we definitely don't want that! It would produce lovely metrics but a terrible model.

### Normalising non utf-8 characters

In [8]:
import unicodedata

def normalise(s):
    return (unicodedata.normalize("NFKD", str(s))
            .encode("ascii", "ignore")
            .decode("ascii"))

X = df.drop(["Label", "Destination Port"], axis=1)
y = df["Label"].apply(normalise)

In [9]:
y.value_counts()

Label
BENIGN                       2095051
DoS Hulk                      172846
DDoS                          128014
PortScan                       90694
DoS GoldenEye                  10286
FTP-Patator                     5931
DoS slowloris                   5385
DoS Slowhttptest                5228
SSH-Patator                     3219
Bot                             1948
Web Attack  Brute Force         1470
Web Attack  XSS                  652
Infiltration                      36
Web Attack  Sql Injection         21
Heartbleed                        11
Name: count, dtype: int64

### Data splitting

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42)
X_test, X_cv, y_test, y_cv       = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42)

In [12]:
X_train.shape[0] + X_test.shape[0] + X_cv.shape[0] == df.shape[0]

True

### Scaling

In [13]:
from sklearn.preprocessing import StandardScaler

In [14]:
std_scaler = StandardScaler().fit(X_train)
X_train = std_scaler.transform(X_train)
X_cv    = std_scaler.transform(X_cv)
X_test  = std_scaler.transform(X_test)

In [15]:
X_train.shape

(1512475, 77)

## Imbalance experiment
This dataset is super imbalanced. There are >2 million benign flows, and some attack categories are in the hundreds of thousands. Wheras Heartbleed, Sql injection, and Infiltration attack classes have less than 100 examples.
We will compare three methods for handling imbalance:
- Modifying class weights;
- Undersampling the majority;
- SMOTE.

These will be evaluated mostly factoring macro-recall, as our aim is to minimise missed attacks at the cost of reduced precision (an increase in false positives).

This will be a fair test, controlling for:
- train, test and CV split (these sets will be constant);
- scaling and other pre-processing;
- choose of model (XGBoost);
- hyper-parameters. To reduce computation cost while maintaining a fair comparison, all experiments will use identical hyper-parameters with a learning rate of 0.2 and a max 300 estimators.

In [16]:
hyper_params = {"n_estimators": 300, "learning_rate": 0.2, "verbosity": 1,
             "random_state": RANDOM_STATE,  "early_stopping_rounds": 20, "tree_method": "hist"}
eval_set = [(X_cv, y_cv)]

## 1. Baseline / Control
We must first establish a baseline to compare the 3 techniques to. Hopefully they won't all just be worse than the baseline!

In [17]:
xgb_model = XGBClassifier(**hyper_params)
xgb_model.fit(X_train, y_train, eval_set = eval_set)

preds = xgb_model.predict(X_test)
evaluate(preds, y_test, le.classes_, "baseline")

del xgb_model, preds

[0]	validation_0-mlogloss:0.35420
[1]	validation_0-mlogloss:0.31449
[2]	validation_0-mlogloss:0.27689
[3]	validation_0-mlogloss:0.23424
[4]	validation_0-mlogloss:0.29670
[5]	validation_0-mlogloss:0.19176
[6]	validation_0-mlogloss:0.29114
[7]	validation_0-mlogloss:0.11610
[8]	validation_0-mlogloss:0.16788
[9]	validation_0-mlogloss:0.17991
[10]	validation_0-mlogloss:0.26047
[11]	validation_0-mlogloss:0.07305
[12]	validation_0-mlogloss:0.07641
[13]	validation_0-mlogloss:0.09154
[14]	validation_0-mlogloss:0.05952
[15]	validation_0-mlogloss:0.05981
[16]	validation_0-mlogloss:0.06873
[17]	validation_0-mlogloss:0.08359
[18]	validation_0-mlogloss:0.04793
[19]	validation_0-mlogloss:0.04470
[20]	validation_0-mlogloss:0.03838
[21]	validation_0-mlogloss:0.03030
[22]	validation_0-mlogloss:0.02790
[23]	validation_0-mlogloss:0.02576
[24]	validation_0-mlogloss:0.02425
[25]	validation_0-mlogloss:0.02321
[26]	validation_0-mlogloss:0.02219
[27]	validation_0-mlogloss:0.02151
[28]	validation_0-mlogloss:0.0

/home/village-spearman/Documents/ML-IDS/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/village-spearman/Documents/ML-IDS/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/village-spearman/Documents/ML-IDS/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

## 2. Sample weights
We will now use the XGBoost `sample_weight` parameter to weigh attack classes with a greater cost proportional to how rare they are. This can be automatically calculated with sklearn `compute_sample_weight` with "balanced" mode which adjusts weights inversely proportional to class frequencies. We will calculate our sample weights ONLY on the training set to avoid leakage.

In [30]:
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight("balanced", y_train)

xgb_sw = XGBClassifier(**hyper_params)
xgb_sw.fit(X_train, y_train, eval_set = eval_set, sample_weight=sample_weights, verbose=0)

preds_sw = xgb_sw.predict(X_test)
evaluate(preds_sw, y_test, le.classes_, "sample weights")

del xgb_sw, preds_sw

--- sample weights ---
accuracy: 0.9981
                           precision    recall  f1-score   support

                   BENIGN       1.00      1.00      1.00    418794
                      Bot       0.55      0.99      0.71       403
                     DDoS       1.00      1.00      1.00     25716
            DoS GoldenEye       0.99      1.00      1.00      2055
                 DoS Hulk       1.00      1.00      1.00     34562
         DoS Slowhttptest       0.95      0.99      0.97      1042
            DoS slowloris       0.99      0.99      0.99      1016
              FTP-Patator       1.00      1.00      1.00      1185
               Heartbleed       0.75      1.00      0.86         3
             Infiltration       0.80      0.80      0.80         5
                 PortScan       0.99      1.00      0.99     18321
              SSH-Patator       1.00      1.00      1.00       639
  Web Attack  Brute Force       0.68      0.73      0.71       280
Web Attack  Sql Injec

## 3. Undersampling
There are two classes of undersampling algorithms, **generation** and **selection**. They both involve creating a new set S' from an original set S, where |S'| < |S|. However, in generation S' ⊄ S wheras in selection S' ⊂ S. 

For the sake of fairness, we will attempt to use a common under-sampling strategy. This will be to tune the majority classes down to have a frequency equal to the third quartile (median would remove far too m

### 3.1 Generation
For generation we will use the cluster centroids method from imbalanced learn, which uses K-means reduce a majority class to the centroids of K-means. Because cluster centroids is a very computationally demanding method, I will first do a trial run with 50k entries. This will let me see if a threshold of the frequency at the third quartile is reasonable and tractable.

In [31]:
import time
from collections import Counter
from sklearn.cluster import MiniBatchKMeans
from imblearn.under_sampling import ClusterCentroids

X_small, y_small = X_train[:50000], y_train[:50000]

counts = Counter(y_train)
vals = list(counts.values())
T_under = int(np.percentile(vals, 75))   # Q3
time_counts = Counter(y_small)
time_strategy = {c: T_under for c, n in counts.items() if n > T_under}

In [32]:
# cc = ClusterCentroids(
#     sampling_strategy=time_strategy,
#     estimator=MiniBatchKMeans(n_init=3, batch_size=4096, verbose=1),
#     random_state=42,
# )

# t = time.time()
# cc.fit_resample(X_small, y_small)
# print(f"{time.time()-t:.1f}s on 50k rows")

# del cc

Way too long! 445.6s (>7 minutes) to converge on a fraction of the dataset. For the cluster centroids method I suppose using the median frequency rather than the third quartline is necessary. This may cause it to not be a fair test but the test is analysing imbalance methods with a broad stroke.
I will try again, this time using the median.

In [33]:
# T_under = int(np.percentile(vals, 50))
# time_strategy = {c: T_under for c, n in counts.items() if n > T_under}

In [34]:
# cc = ClusterCentroids(
#     sampling_strategy=time_strategy,
#     estimator=MiniBatchKMeans(n_init=3, batch_size=4096, verbose=1),
#     random_state=42,
# )

# t = time.time()
# cc.fit_resample(X_small, y_small)
# print(f"{time.time()-t:.1f}s on 50k rows")

# del cc, time_strategy

Much better! This took only 18.9 seconds, saving more than 7 minutes :D

Thus, I will use the median as the threshold in the strategy.

In [35]:
T_cc = int(np.percentile(vals, 50))
cc_strategy = {c: T_cc for c, n in counts.items() if n > T_cc}

cc = ClusterCentroids(sampling_strategy=cc_strategy, 
                      estimator=MiniBatchKMeans(n_init=3, batch_size=4096),
                      random_state=RANDOM_STATE)
X_train_cc, y_train_cc = cc.fit_resample(X_train, y_train)

xgb_cc = XGBClassifier(**hyper_params)
xgb_cc.fit(X_train_cc, y_train_cc, eval_set=eval_set, verbose=0)

preds_cc = xgb_cc.predict(X_test)
evaluate(preds_cc, y_test, le.classes_, "Cluster Centroids (under)")

del cc, xgb_cc, preds_cc

--- Cluster Centroids (under) ---
accuracy: 0.9206
                           precision    recall  f1-score   support

                   BENIGN       1.00      0.91      0.95    418794
                      Bot       0.06      1.00      0.12       403
                     DDoS       0.95      1.00      0.97     25716
            DoS GoldenEye       0.38      1.00      0.55      2055
                 DoS Hulk       0.93      0.99      0.96     34562
         DoS Slowhttptest       0.22      0.99      0.36      1042
            DoS slowloris       0.37      0.99      0.54      1016
              FTP-Patator       0.60      1.00      0.75      1185
               Heartbleed       0.75      1.00      0.86         3
             Infiltration       0.09      0.80      0.15         5
                 PortScan       0.97      1.00      0.98     18321
              SSH-Patator       0.21      1.00      0.35       639
  Web Attack  Brute Force       0.07      0.85      0.13       280
Web Attack

### 3.2 Selection
Selection is split into 2 further groups: Controlled an Cleaning. Controlled involves reducing majority classes down to an arbitrary user-specified amount (we will use third quartile as above), and cleaning involves using an algorithm to remove observations that follow a certain criteria, thus the amount of samples at the end can not be user-specified. 

Cleaning  algorithms are computationally expensive, and for this reason we will combine controlled and cleaning methods into one pipeline. Controlled selection will vastly reduce the load fed into the cleaning section.

In [36]:
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler, EditedNearestNeighbours

T_rus = int(np.percentile(vals, 50))
rus_strategy = {c: T_rus for c, n in counts.items() if n > T_rus}

pipe = Pipeline([
    ('rus', RandomUnderSampler(sampling_strategy=rus_strategy, random_state=RANDOM_STATE)), 
    ('enn', EditedNearestNeighbours(n_jobs=-1)),
    ('clf', XGBClassifier(**hyper_params)),
])
pipe.fit(X_train, y_train, clf__eval_set=eval_set)

preds_sel = pipe.predict(X_test)
evaluate(preds_sel, y_test, le.classes_, "random -> ENN")

[0]	validation_0-mlogloss:1.61579
[1]	validation_0-mlogloss:1.32929
[2]	validation_0-mlogloss:1.13205
[3]	validation_0-mlogloss:0.96488
[4]	validation_0-mlogloss:0.80622
[5]	validation_0-mlogloss:0.68147
[6]	validation_0-mlogloss:0.58533
[7]	validation_0-mlogloss:0.50329
[8]	validation_0-mlogloss:0.43661
[9]	validation_0-mlogloss:0.38438
[10]	validation_0-mlogloss:0.34086
[11]	validation_0-mlogloss:0.30430
[12]	validation_0-mlogloss:0.27408
[13]	validation_0-mlogloss:0.24270
[14]	validation_0-mlogloss:0.21632
[15]	validation_0-mlogloss:0.19379
[16]	validation_0-mlogloss:0.17651
[17]	validation_0-mlogloss:0.16084
[18]	validation_0-mlogloss:0.14676
[19]	validation_0-mlogloss:0.13406
[20]	validation_0-mlogloss:0.12550
[21]	validation_0-mlogloss:0.11796
[22]	validation_0-mlogloss:0.11118
[23]	validation_0-mlogloss:0.10504
[24]	validation_0-mlogloss:0.09956
[25]	validation_0-mlogloss:0.09589
[26]	validation_0-mlogloss:0.09234
[27]	validation_0-mlogloss:0.08966
[28]	validation_0-mlogloss:0.0

## 4. Oversampling

We will try two methods of oversampling:
- Naive random,
- SMOTE.

These will both be using methods from the imbalanced learn library. For the sake of fairness, the strategy will be to bring each class to have a frequency equal to the median frequency (of the training set). So, if the median is 5,000, the classes with the least frequent half will be oversampled up to 5,000 entires.

In [37]:
from collections import Counter
counts = Counter(y_train)
T = int(np.median(list(counts.values())))
k = 5
strategy = {c: T for c, n in counts.items() if k < n < T}

### 4.1 Naive Random
The simplest oversampling method would be naive random - sampling with replacement. This would allow rows of minority classes to be duplicated multiple times. While this could potentially help some rare classes (likely would not help classes with ~10 entries), it is functionally the same as changing the weight of the class from (2). 

Despite this, we will still trial the method.

In [38]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=RANDOM_STATE, sampling_strategy=strategy)
X_train_r, y_train_r = ros.fit_resample(X_train, y_train)

xgb_ros = XGBClassifier(**hyper_params)
xgb_ros.fit(X_train_r, y_train_r, eval_set = eval_set, verbose=1)

preds_ros = xgb_ros.predict(X_test)
evaluate(preds_ros, y_test, le.classes_, "Naive oversampling (median)")

del ros, xgb_ros, preds_ros

[0]	validation_0-mlogloss:0.33242
[1]	validation_0-mlogloss:0.26834
[2]	validation_0-mlogloss:0.27583
[3]	validation_0-mlogloss:0.21507
[4]	validation_0-mlogloss:0.20808
[5]	validation_0-mlogloss:0.19683
[6]	validation_0-mlogloss:0.15316
[7]	validation_0-mlogloss:0.25622
[8]	validation_0-mlogloss:0.09166
[9]	validation_0-mlogloss:0.11494
[10]	validation_0-mlogloss:0.07186
[11]	validation_0-mlogloss:0.06230
[12]	validation_0-mlogloss:0.07614
[13]	validation_0-mlogloss:0.04525
[14]	validation_0-mlogloss:0.04194
[15]	validation_0-mlogloss:0.05411
[16]	validation_0-mlogloss:0.05259
[17]	validation_0-mlogloss:0.03879
[18]	validation_0-mlogloss:0.03449
[19]	validation_0-mlogloss:0.05154
[20]	validation_0-mlogloss:0.03227
[21]	validation_0-mlogloss:0.03166
[22]	validation_0-mlogloss:0.02910
[23]	validation_0-mlogloss:0.02828
[24]	validation_0-mlogloss:0.02630
[25]	validation_0-mlogloss:0.02433
[26]	validation_0-mlogloss:0.02379
[27]	validation_0-mlogloss:0.02189
[28]	validation_0-mlogloss:0.0

### 4.2 SMOTE - synthetic minority oversampling technique

Creates entries in rare classes by interpolating between 2 actually existing entries. For the sake of the test, I will allow very small classes to also be SMOTE'd. Even classes which have <10 entries. It will be interesting to see how the model changes when it recieves thousands of synthetic data points.

In [39]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(sampling_strategy=strategy, k_neighbors=k)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

xgb_sm = XGBClassifier(**hyper_params)
xgb_sm.fit(X_train_sm, y_train_sm, eval_set = eval_set, verbose=0)

preds_sm = xgb_sm.predict(X_test)
evaluate(preds_sm, y_test, le.classes_, "SMOTE (median)")

del xgb_sm, preds_sm

--- SMOTE (median) ---
accuracy: 0.9973
                           precision    recall  f1-score   support

                   BENIGN       1.00      1.00      1.00    418794
                      Bot       0.72      0.88      0.79       403
                     DDoS       1.00      1.00      1.00     25716
            DoS GoldenEye       0.99      0.98      0.99      2055
                 DoS Hulk       1.00      1.00      1.00     34562
         DoS Slowhttptest       0.88      0.98      0.93      1042
            DoS slowloris       0.99      0.99      0.99      1016
              FTP-Patator       1.00      0.99      1.00      1185
               Heartbleed       1.00      1.00      1.00         3
             Infiltration       0.17      0.80      0.29         5
                 PortScan       0.99      1.00      0.99     18321
              SSH-Patator       0.99      0.99      0.99       639
  Web Attack  Brute Force       0.54      0.59      0.56       280
Web Attack  Sql Injec

## 5. synthesis
random undersampling -> SMOTE -> ENN
random understampling and SMOTE will work on mutually exclusive classes - the sets left and right of the median. 

In [40]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler, EditedNearestNeighbours
from imblearn.pipeline import Pipeline

T = int(np.percentile(vals, 50))
rus_strategy = {c: T for c, n in counts.items() if n > T}
smt_strategy = {c: T for c, n in counts.items() if n < T}

pipe = Pipeline([
    ('rus', RandomUnderSampler(sampling_strategy=rus_strategy, random_state=RANDOM_STATE)), 
    ('smt', SMOTE(sampling_strategy=strategy, k_neighbors=k)),
    ('enn', EditedNearestNeighbours(n_jobs=-1)),
    ('clf', XGBClassifier(**hyper_params)),
])
pipe.fit(X_train, y_train, clf__eval_set=eval_set)


[0]	validation_0-mlogloss:1.77590
[1]	validation_0-mlogloss:1.37346
[2]	validation_0-mlogloss:1.10657
[3]	validation_0-mlogloss:0.92000
[4]	validation_0-mlogloss:0.76835
[5]	validation_0-mlogloss:0.65159
[6]	validation_0-mlogloss:0.55338
[7]	validation_0-mlogloss:0.47496
[8]	validation_0-mlogloss:0.40988
[9]	validation_0-mlogloss:0.35570
[10]	validation_0-mlogloss:0.31487
[11]	validation_0-mlogloss:0.27906
[12]	validation_0-mlogloss:0.24977
[13]	validation_0-mlogloss:0.22493
[14]	validation_0-mlogloss:0.20184
[15]	validation_0-mlogloss:0.18253
[16]	validation_0-mlogloss:0.16184
[17]	validation_0-mlogloss:0.14579
[18]	validation_0-mlogloss:0.13129
[19]	validation_0-mlogloss:0.12041
[20]	validation_0-mlogloss:0.10946
[21]	validation_0-mlogloss:0.10168
[22]	validation_0-mlogloss:0.09418
[23]	validation_0-mlogloss:0.08793
[24]	validation_0-mlogloss:0.08212
[25]	validation_0-mlogloss:0.07725
[26]	validation_0-mlogloss:0.07275
[27]	validation_0-mlogloss:0.06887
[28]	validation_0-mlogloss:0.0

,steps,"[('rus', ...), ('smt', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](15,)","[ 0, 1, 2,...,12,13,14]"
n_features_in_,int,77
,sampling_strategy,"{np.int64(0): 3162, np.int64(2): 3162, np.int64(3): 3162, np.int64(4): 3162, ...}"
,random_state,5
,replacement,False
Name,Type,Value


In [41]:
preds_syn = pipe.predict(X_test)
evaluate(preds_syn, y_test, le.classes_, "random -> SMOTE -> ENN")

--- random -> SMOTE -> ENN ---
accuracy: 0.9914
                           precision    recall  f1-score   support

                   BENIGN       1.00      0.99      1.00    418794
                      Bot       0.17      1.00      0.29       403
                     DDoS       0.99      1.00      1.00     25716
            DoS GoldenEye       0.94      1.00      0.97      2055
                 DoS Hulk       1.00      0.99      0.99     34562
         DoS Slowhttptest       0.85      0.99      0.92      1042
            DoS slowloris       0.75      0.99      0.86      1016
              FTP-Patator       0.89      1.00      0.94      1185
               Heartbleed       0.75      1.00      0.86         3
             Infiltration       0.03      0.80      0.05         5
                 PortScan       0.99      0.99      0.99     18321
              SSH-Patator       0.95      1.00      0.97       639
  Web Attack  Brute Force       0.44      0.62      0.51       280
Web Attack  S